# Cache: гранаты, внешние эталоны и наблюдаемый эффект

## Краткий результат

Пространственные агрегаты охватывают **10 000 матчей и 2 385 417 бросков**.
Полные гранатные профили содержат **99 921 пару «игрок × матч»**.
Подробное сравнение с десятью внешними раскидками использует 167 499 ближайших бросков.

Положительная связь близости к эталону с эффектом не подтверждена для всего выбранного пула.
Flash-модель показывает умеренный сигнал: ROC-AUC test ≈ **0,615**.
Связь с независимым КПД и победами пока не рассчитана: нужных исходных полей нет.


## Контекст и метод

`ml_algorithm.py` выполняет расчёты, `visualizations.py` содержит функции построения,
а этот ноутбук показывает результаты последовательно под ячейками.
Все три файла и `dataset.sqlite` должны лежать рядом, в ядре из каталога проекта.


In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from ml_algorithm import database, read_document
from visualizations import (
    draw_associations,
    draw_correlations,
    draw_enemy_example,
    draw_grid,
    draw_model,
)

DATA = Path("dataset.sqlite").resolve()  # Можно указать копию базы.
if not DATA.is_file():
    raise FileNotFoundError("Откройте ноутбук из папки, содержащей dataset.sqlite.")

PLOT_STYLE = {
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
}

## Данные

Читаются готовые счётчики и результаты из SQLite. Исходный pickle размером 1,57 ГБ
не открывается. Пространственный обзор и полные профили имеют больший охват,
чем выборка сопоставления с эталонами и выборка flash-модели.

In [ ]:
with database(DATA) as connection:
    grid = [dict(row) for row in connection.execute("SELECT * FROM spatial_grid")]
    documents = {
        key: read_document(connection, key)
        for key in (
            "extraction", "references", "examples", "associations",
            "flash_model", "flash_predictions", "outcomes",
            "round_predictions", "player_profiles",
        )
    }
    nearby_count, nearby_matches = connection.execute(
        "SELECT COUNT(*), COUNT(DISTINCT match_id) FROM throws"
    ).fetchone()

references = documents["references"]["references"]
extraction = documents["extraction"]
profiles = documents["player_profiles"]
flash = documents["flash_model"]
display(Markdown(
    "| Уровень | Наблюдения | Матчи |\n|---|---:|---:|\n"
    f"| Все броски: пространственные агрегаты | {extraction['counts']['throws']:,} | {extraction['matches']:,} |\n"
    f"| Все гранатные профили: игрок × матч | {profiles['player_matches']:,} | {profiles['matches']:,} |\n"
    f"| Броски рядом с эталонами | {nearby_count:,} | {nearby_matches:,} |\n"
    f"| Flash-модель: train | {flash['train']['rows']:,} | {flash['train']['matches']:,} |\n"
    f"| Flash-модель: test | {flash['test']['rows']:,} | {flash['test']['matches']:,} |"
))

## Результаты

### 1. Позиции бросков за T и CT

Общая цветовая шкала позволяет сопоставлять плотность событий.
Ячейки имеют размеры 256 × 256 × 64 игровых единицы. Звёздами показаны
внешние позиции подготовки; стены и траектории по этой сетке не восстанавливаются.

In [ ]:
with plt.rc_context(PLOT_STYLE):
    figure = draw_grid(grid, extraction, references)
    display(figure)
    plt.close(figure)

### 2. Соответствие эталону и ослепление противника

Сравниваются неподвижные броски стоя с известным исходом, внутри и вне допуска.
Доверительные интервалы получены пересэмплированием целых матчей.
Количество матчей справа относится к конкретному эталону и фильтру.

In [ ]:
with plt.rc_context(PLOT_STYLE):
    figure = draw_associations(documents["associations"], references)
    display(figure)
    plt.close(figure)

Для Truck Wall на CT наблюдается отрицательная связь, для B Main интервал
включает ноль. Для Quad внутри допуска слишком мало матчей для интервала.
Это не доказывает причинный вред или пользу самих рецептов:
контекст действий и совместимость версий остаются непроверенными.

### 3. Положение противников в отдельном эпизоде

Проекция XY с высотами Z показывает позиции целей с зарегистрированным ослеплением.
Это один выбранный иллюстративный бросок, а не выборка для оценки общей связи.
Отрезок между началом и концом полёта не является восстановленной траекторией.

In [ ]:
with plt.rc_context(PLOT_STYLE):
    figure = draw_enemy_example(documents["examples"], references)
    display(figure)
    plt.close(figure)

### 4. Проверка flash-модели на отдельных матчах

Модель оценивает зарегистрированное ослепление по признакам момента броска.
Матчи train и test не пересекаются. ROC-AUC оценивает разделение классов,
калибровка сопоставляет предсказанную вероятность с наблюдаемой долей.
Постоянный прогноз определяется долей положительных исходов только в train.

In [ ]:
with plt.rc_context(PLOT_STYLE):
    if flash["status"] == "trained":
        assert not set(flash["train_match_ids"]) & set(flash["test_match_ids"])
        figure = draw_model(
            flash, documents["flash_predictions"],
            "Флешки из пула: эффект по признакам момента броска",
        )
        display(figure)
        plt.close(figure)
    else:
        display(Markdown("Модель не обучена: " + flash.get("reason", "нет результата")))

In [ ]:
if flash["status"] == "trained":
    display(Markdown(
        f"**ROC-AUC test: {flash['test']['metrics']['roc_auc']:.4f}.** "
        f"Brier модели: {flash['test']['metrics']['brier']:.4f}; "
        f"постоянного прогноза: {flash['baseline']['brier']:.4f} (меньше — лучше). "
        "Сигнал умеренный; результат относится к известным исходам флешек рядом с пулом эталонов."
    ))

### 5. Независимый КПД и победы в раундах

Ниже выводится статус доступности данных. Отсутствие результата здесь не является
ошибкой выполнения остальных ячеек. Добавление полного гранатного профиля не создаёт
независимую оценку общей игры или отсутствующие исходы раундов.

In [ ]:
outcomes = documents["outcomes"]
with plt.rc_context(PLOT_STYLE):
    if outcomes["status"] == "context_loaded":
        figure = draw_correlations(outcomes)
        if figure is not None:
            display(figure)
            plt.close(figure)
        if outcomes["round_model"]["status"] == "trained":
            figure = draw_model(
                outcomes["round_model"], documents["round_predictions"],
                "Победа: ретроспективная модель по данным раунда",
            )
            display(figure)
            plt.close(figure)
    else:
        display(Markdown(
            "**Статус: дополнительные данные отсутствуют.** "
            "В исходном датасете нет победителей раундов и независимого КПД общей игры. "
            "Соответствующие корреляции не рассчитаны. Формат внешнего CSV описан в техническом README."
        ))

## Выводы и ограничения

Пространство событий, сопоставление с внешними рецептами и наблюдаемое ослепление
воспроизводимо визуализируются. Положительная связь близости к рецепту с эффектом
в целом не подтверждена; ML-проверка даёт умеренный сигнал на ограниченной выборке.

Тактическая успешность дыма, причинное влияние гранат на победу, полноценная траектория
и независимый КПД не восстанавливаются из доступных полей.
Определения, источники эталонов, допуски и ограничения подробно описаны в `README_CONCEPT.md`.